# 🏺 Tourasna — Full Pipeline Test + FastAPI (Stage 4)
# Detection → Classification → Translation → API
# Repo: github.com/Tourasna/Tourasna

In [4]:
# ── Check translator files ────────────────────────────
import os
path = '/content/drive/MyDrive/Tourasna/hieroglyphics/translator'
if os.path.exists(path):
    for f in os.listdir(path):
        size = os.path.getsize(f'{path}/{f}')
        print(f"  {f} ({size/1024:.0f} KB)")
else:
    print(f"Path not found: {path}")
    # Search for it
    !find /content/drive/MyDrive/Tourasna -name "src_vocab.json" 2>/dev/null
    !find /content/drive/MyDrive/Tourasna -name "best_translator.pth" 2>/dev/null

Path not found: /content/drive/MyDrive/Tourasna/hieroglyphics/translator


In [5]:
# ── Search for translator files ───────────────────────
!find /content/drive/MyDrive -name "best_translator.pth" 2>/dev/null
!find /content/drive/MyDrive -name "src_vocab.json" 2>/dev/null
!find /content/drive/MyDrive -name "tgt_bpe.model" 2>/dev/null

In [6]:
# ── Cell 2: Load all 3 models ────────────────────────
!pip install ultralytics sentencepiece -q

from google.colab import drive
drive.mount('/content/drive')

import torch, json, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ── 1. Detection Model (YOLOv8) ──────────────────────
from ultralytics import YOLO
detector = YOLO('/content/drive/MyDrive/Tourasna/hieroglyphics/detector/yolo_run/weights/best.pt')
print("✅ Detector loaded")

# ── 2. Classification Model (EfficientNetB0) ─────────
from torchvision import models
import torch.nn as nn

# Load metadata
with open('/content/drive/MyDrive/Tourasna/hieroglyphics/classifier/classifier_metadata.json', 'r') as f:
    cls_meta = json.load(f)

class_names = cls_meta['class_names']
NUM_CLASSES = cls_meta['num_classes']

# Rebuild model
classifier = models.efficientnet_b0(weights=None)
classifier.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(classifier.classifier[1].in_features, NUM_CLASSES)
)
classifier.load_state_dict(torch.load(
    '/content/drive/MyDrive/Tourasna/hieroglyphics/classifier/best_classifier.pth',
    map_location=device
))
classifier = classifier.to(device)
classifier.eval()
print(f"✅ Classifier loaded ({NUM_CLASSES} classes)")

# ── 3. Translation Model (Transformer) ───────────────
import sentencepiece as spm
import math

# Load tokenizers
with open('/content/drive/MyDrive/Tourasna/hieroglyphics/translator/src_vocab.json', 'r') as f:
    vocab_data = json.load(f)
src_token2idx = vocab_data['src_token2idx']
src_vocab = vocab_data['src_vocab']

tgt_sp = spm.SentencePieceProcessor()
tgt_sp.load('/content/drive/MyDrive/Tourasna/hieroglyphics/translator/tgt_bpe.model')

# Rebuild translator model
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=300):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class HieroglyphTranslator(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, nhead=8,
                 num_encoder_layers=4, num_decoder_layers=4, dim_ff=1024, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=0)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model, dropout, max_len=300)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_ff, dropout=dropout, batch_first=True
        )
        self.output_proj = nn.Linear(d_model, tgt_vocab_size)
    def forward(self, src, tgt):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(src.device)
        src_padding_mask = (src == 0)
        tgt_padding_mask = (tgt == 0)
        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.tgt_embedding(tgt) * math.sqrt(self.d_model))
        output = self.transformer(src_emb, tgt_emb, tgt_mask=tgt_mask,
                                  src_key_padding_mask=src_padding_mask,
                                  tgt_key_padding_mask=tgt_padding_mask)
        return self.output_proj(output)

translator = HieroglyphTranslator(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=tgt_sp.get_piece_size(),
    d_model=256, nhead=8,
    num_encoder_layers=4, num_decoder_layers=4,
    dim_ff=1024, dropout=0.15
).to(device)

translator.load_state_dict(torch.load(
    '/content/drive/MyDrive/Tourasna/hieroglyphics/translator/best_translator.pth',
    map_location=device
))
translator.eval()
print(f"✅ Translator loaded")

print("\n🎉 All 3 models ready!")

Mounted at /content/drive
Device: cpu
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Detector loaded
✅ Classifier loaded (117 classes)
✅ Translator loaded

🎉 All 3 models ready!


In [7]:
# ── Cell 3: Full Pipeline Function ────────────────────
from PIL import Image
from torchvision import transforms
import torch.nn.functional as F

# ── Classifier transform ──────────────────────────────
classify_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ── Beam search translate ─────────────────────────────
def translate_beam(src_text, beam_size=5, max_len=100):
    translator.eval()
    src_tokens = src_text.split()[:200]
    src_ids = [src_token2idx.get(t, src_token2idx['<unk>']) for t in src_tokens]
    src_tensor = torch.tensor([src_ids]).to(device)
    beams = [(0.0, [1])]
    completed = []
    for _ in range(max_len):
        candidates = []
        for score, tokens in beams:
            if tokens[-1] == 2:
                completed.append((score, tokens))
                continue
            tgt_tensor = torch.tensor([tokens]).to(device)
            with torch.no_grad():
                output = translator(src_tensor, tgt_tensor)
            log_probs = F.log_softmax(output[0, -1], dim=-1)
            top_probs, top_ids = log_probs.topk(beam_size)
            for prob, idx in zip(top_probs, top_ids):
                token_id = idx.item()
                penalty = 0.0
                if len(tokens) >= 2 and token_id == tokens[-1]:
                    penalty = -2.0
                if len(tokens) >= 3 and token_id == tokens[-1] == tokens[-2]:
                    penalty = -5.0
                candidates.append((score + prob.item() + penalty, tokens + [token_id]))
        if not candidates:
            break
        candidates.sort(key=lambda x: x[0] / max(len(x[1]), 1), reverse=True)
        beams = candidates[:beam_size]
    all_results = completed + beams
    if not all_results:
        return ""
    all_results.sort(key=lambda x: x[0] / max(len(x[1]), 1), reverse=True)
    best_tokens = all_results[0][1]
    clean = [t for t in best_tokens if t not in [0, 1, 2]]
    return tgt_sp.decode(clean)

# ── Full Pipeline ─────────────────────────────────────
def translate_inscription(image_path):
    """
    Full pipeline: Image → Detection → Classification → Translation
    """
    print(f"📸 Processing: {image_path}")
    print("=" * 60)

    # ── Stage 1: Detection ────────────────────────────
    results = detector(image_path, conf=0.3, verbose=False)
    boxes = results[0].boxes

    glyph_boxes = []
    for box in boxes:
        cls_id = int(box.cls[0])
        cls_name = detector.names[cls_id]
        conf = float(box.conf[0])
        bbox = box.xyxy[0].tolist()
        if cls_name == 'glyph':
            glyph_boxes.append({'bbox': bbox, 'conf': conf})

    print(f"🔍 Stage 1 — Detection: Found {len(glyph_boxes)} glyphs")

    if not glyph_boxes:
        print("❌ No glyphs detected!")
        return None

    # Sort left-to-right, top-to-bottom (reading order)
    glyph_boxes.sort(key=lambda b: (round(b['bbox'][1] / 50), b['bbox'][0]))

    # ── Stage 2: Classification ───────────────────────
    img = Image.open(image_path).convert('RGB')
    gardiner_codes = []

    for i, gb in enumerate(glyph_boxes):
        x1, y1, x2, y2 = [int(c) for c in gb['bbox']]
        # Add small padding
        pad = 5
        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)
        x2 = min(img.width, x2 + pad)
        y2 = min(img.height, y2 + pad)

        crop = img.crop((x1, y1, x2, y2))
        tensor = classify_transform(crop).unsqueeze(0).to(device)

        with torch.no_grad():
            output = classifier(tensor)
            prob = F.softmax(output, dim=1)
            conf, pred = prob.max(1)

        code = class_names[pred.item()]
        gardiner_codes.append({
            'code': code,
            'confidence': round(conf.item(), 3),
            'bbox': gb['bbox']
        })

    codes_str = ' '.join([g['code'] for g in gardiner_codes])
    print(f"🏷️  Stage 2 — Classification: {codes_str}")

    # ── Stage 3: Translation ──────────────────────────
    translation = translate_beam(codes_str, beam_size=5)
    print(f"📝 Stage 3 — Translation: {translation}")

    # ── Result ────────────────────────────────────────
    result = {
        'num_glyphs': len(gardiner_codes),
        'gardiner_codes': gardiner_codes,
        'gardiner_sequence': codes_str,
        'translation': translation,
    }

    print(f"\n✅ Pipeline complete!")
    return result

In [11]:
# ── Cell 4: Test with an image ────────────────────────
# Upload a hieroglyphic image
from google.colab import files
print("📤 Upload a hieroglyphic inscription image:")
uploaded = files.upload()

# Run pipeline on uploaded image
for filename in uploaded:
    result = translate_inscription(filename)
    if result:
        print(f"\n{'=' * 60}")
        print(f"📊 FULL RESULT:")
        print(json.dumps(result, indent=2, ensure_ascii=False))

📤 Upload a hieroglyphic inscription image:


Saving Egypt_Hieroglyphe4.webp to Egypt_Hieroglyphe4 (1).webp
📸 Processing: Egypt_Hieroglyphe4 (1).webp
🔍 Stage 1 — Detection: Found 7 glyphs
🏷️  Stage 2 — Classification: S29 I9 G17 S29 D21 P98 Y5
📝 Stage 3 — Translation: The head of the linen stuff, head of the two barns, head of the two barns, head of the two arms, head of the two arms, head of the two arms, head of the two arms, head of the two arms, head of the two arms, head of the two arms, head of the two arms, head of the two arms, head of the two arms, head of the two arms, head of the two arms, head of the two arms, head of his

✅ Pipeline complete!

📊 FULL RESULT:
{
  "num_glyphs": 7,
  "gardiner_codes": [
    {
      "code": "S29",
      "confidence": 0.983,
      "bbox": [
        56.192138671875,
        61.551513671875,
        89.74858856201172,
        162.44589233398438
      ]
    },
    {
      "code": "I9",
      "confidence": 0.994,
      "bbox": [
        93.96162414550781,
        64.63607025146484,
        192

In [12]:
# ── Cell 5: Improved Pipeline — Higher confidence ─────
def translate_inscription_v2(image_path, det_conf=0.5, cls_conf=0.6):
    """
    Improved pipeline with confidence filtering
    """
    print(f"📸 Processing: {image_path}")
    print("=" * 60)

    # ── Stage 1: Detection (higher threshold) ─────────
    results = detector(image_path, conf=det_conf, verbose=False)
    boxes = results[0].boxes

    glyph_boxes = []
    for box in boxes:
        cls_id = int(box.cls[0])
        cls_name = detector.names[cls_id]
        conf = float(box.conf[0])
        bbox = box.xyxy[0].tolist()
        if cls_name == 'glyph':
            glyph_boxes.append({'bbox': bbox, 'conf': conf})

    print(f"🔍 Detection: Found {len(glyph_boxes)} glyphs (conf > {det_conf})")

    if not glyph_boxes:
        print("❌ No glyphs detected!")
        return None

    # Sort reading order: top-to-bottom, left-to-right
    glyph_boxes.sort(key=lambda b: (round(b['bbox'][1] / 50), b['bbox'][0]))

    # ── Stage 2: Classification (filter low confidence) ─
    img = Image.open(image_path).convert('RGB')
    gardiner_codes = []

    for gb in glyph_boxes:
        x1, y1, x2, y2 = [int(c) for c in gb['bbox']]
        pad = 5
        x1, y1 = max(0, x1-pad), max(0, y1-pad)
        x2, y2 = min(img.width, x2+pad), min(img.height, y2+pad)

        crop = img.crop((x1, y1, x2, y2))
        tensor = classify_transform(crop).unsqueeze(0).to(device)

        with torch.no_grad():
            output = classifier(tensor)
            prob = F.softmax(output, dim=1)
            conf, pred = prob.max(1)

        code = class_names[pred.item()]
        c = round(conf.item(), 3)

        if c >= cls_conf:  # Only keep high-confidence classifications
            gardiner_codes.append({'code': code, 'confidence': c, 'bbox': gb['bbox']})
            print(f"   ✅ {code} (conf: {c})")
        else:
            print(f"   ❌ {code} (conf: {c}) — skipped")

    if not gardiner_codes:
        print("❌ No confident classifications!")
        return None

    codes_str = ' '.join([g['code'] for g in gardiner_codes])
    print(f"\n🏷️  Gardiner sequence: {codes_str}")

    # ── Stage 3: Translation ──────────────────────────
    translation = translate_beam(codes_str, beam_size=5, max_len=50)
    print(f"📝 Translation: {translation}")

    return {
        'num_glyphs': len(gardiner_codes),
        'gardiner_codes': gardiner_codes,
        'gardiner_sequence': codes_str,
        'translation': translation,
    }

# ── Re-test with the same image ──────────────────────
result = translate_inscription_v2('Egypt_Hieroglyphe4 (1).webp')

📸 Processing: Egypt_Hieroglyphe4 (1).webp
🔍 Detection: Found 5 glyphs (conf > 0.5)
   ✅ S29 (conf: 0.983)
   ✅ I9 (conf: 0.994)
   ✅ G17 (conf: 0.987)
   ✅ D21 (conf: 0.907)
   ❌ Y5 (conf: 0.405) — skipped

🏷️  Gardiner sequence: S29 I9 G17 D21
📝 Translation: (It) will be given to it (i.e. the patient) to the head of the king's clerks, head of the two arms, head of the two arms, head of the two arms, head of the two arms,
